# 🔗 Calliope AI + Jupyter AI Magic Integration

This notebook demonstrates how Calliope AI wraps and extends Jupyter AI magic commands, giving you the best of both worlds: powerful natural language database queries AND all Jupyter AI capabilities.

## What You'll Learn
- Using Jupyter AI magics alongside Calliope
- Combining AI code generation with database queries
- Multi-model workflows
- Creating data analysis pipelines with AI assistance

## Available Providers
Calliope supports all Jupyter AI providers including:
- **OpenAI** (GPT-4, GPT-3.5)
- **Anthropic** (Claude models)
- **Google** (Gemini)
- **Cohere**, **AI21**, **Mistral**, **Bedrock**, **Ollama**, and more!

## 📋 Check Available Models

In [ ]:
%%calliope list-models

## 🗄️ Setup: Create Sample Database

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

conn = sqlite3.connect('marketing_data.db')

# Campaign data
campaigns = pd.DataFrame({
    'campaign_id': range(1, 21),
    'campaign_name': [f'Campaign_{i}' for i in range(1, 21)],
    'channel': np.random.choice(['Email', 'Social', 'Search', 'Display'], 20),
    'budget': np.random.randint(5000, 50000, 20),
    'start_date': [(datetime(2024, 1, 1) + timedelta(days=np.random.randint(0, 60))).strftime('%Y-%m-%d') for _ in range(20)]
})

# Performance metrics
metrics = pd.DataFrame({
    'metric_id': range(1, 101),
    'campaign_id': np.random.randint(1, 21, 100),
    'date': [(datetime(2024, 1, 1) + timedelta(days=np.random.randint(0, 90))).strftime('%Y-%m-%d') for _ in range(100)],
    'impressions': np.random.randint(1000, 100000, 100),
    'clicks': np.random.randint(10, 5000, 100),
    'conversions': np.random.randint(0, 100, 100),
    'cost': np.random.uniform(100, 2000, 100).round(2)
})

campaigns.to_sql('campaigns', conn, if_exists='replace', index=False)
metrics.to_sql('metrics', conn, if_exists='replace', index=False)

conn.close()
print("✅ Marketing database created!")

In [ ]:
%%calliope add-database
{
    "datasource_id": "marketing",
    "name": "Marketing Analytics",
    "dialect": "sqlite",
    "connection_details": {
        "database": "marketing_data.db"
    }
}

## 🎯 Pattern 1: Query → AI Analysis

Use Calliope to query data, then use Jupyter AI to analyze results:

In [ ]:
%%calliope ask-sql marketing
Show campaign performance by channel: impressions, clicks, conversions, and cost

In [ ]:
%%ai anthropic-chat:claude-3-5-sonnet-20241022
Based on the marketing data shown above, what insights can you provide about campaign performance? 
Which channels are most cost-effective? Provide 3 actionable recommendations.

## 🔄 Pattern 2: AI Code Generation → Execute on Data

In [ ]:
%%ai openai-chat:gpt-4o --format code
Write Python code to calculate ROI (return on investment) for marketing campaigns. 
ROI = (conversions * avg_conversion_value - cost) / cost * 100
Use avg_conversion_value = 150 per conversion

In [ ]:
%%calliope run-sql marketing
SELECT 
    c.campaign_name,
    c.channel,
    SUM(m.conversions) as total_conversions,
    SUM(m.cost) as total_cost,
    ROUND(((SUM(m.conversions) * 150 - SUM(m.cost)) / SUM(m.cost) * 100), 2) as roi_percent
FROM campaigns c
JOIN metrics m ON c.campaign_id = m.campaign_id
GROUP BY c.campaign_id, c.campaign_name, c.channel
ORDER BY roi_percent DESC

## 🎨 Pattern 3: Multi-Model Analysis

Use different AI models for different tasks:

In [ ]:
# Use Anthropic for complex SQL generation
%%calliope ask-sql marketing --sql-model anthropic
Calculate click-through rate (CTR) and conversion rate by channel over time

In [ ]:
# Use OpenAI for strategic recommendations
%%ai openai-chat:gpt-4o
Based on the CTR and conversion rate data above, provide a strategic marketing plan 
to optimize budget allocation across channels

In [ ]:
# Use Gemini for visualization code
%%ai gemini:gemini-2.5-flash --format code
Create a matplotlib visualization showing budget allocation vs ROI for each marketing channel

## 📊 Pattern 4: Automated Reporting Pipeline

In [ ]:
# Step 1: Get key metrics
%%calliope ask-sql marketing --sql-model anthropic
Show total impressions, clicks, conversions, cost, and calculated CTR and CPC by campaign

In [ ]:
# Step 2: Generate executive summary
%%ai anthropic-chat:claude-3-5-sonnet-20241022
Create an executive summary of the marketing performance above. 
Format as:
1. Key Metrics Overview
2. Top Performing Campaigns
3. Areas of Concern
4. Recommendations

## 🔬 Pattern 5: Statistical Analysis with AI

In [ ]:
%%calliope ask-sql marketing --sql-model anthropic
Calculate mean, median, and standard deviation of conversion rates by channel

In [ ]:
%%ai openai-chat:gpt-4o --format code
Write Python code using scipy to perform a statistical significance test 
comparing conversion rates between the Email and Social channels

## 🤖 Pattern 6: General AI Assistant

Use Calliope's `ask` command for non-database questions:

In [ ]:
%%calliope ask marketing --model claude3
What are best practices for A/B testing marketing campaigns?

In [ ]:
%%calliope ask marketing --model gpt4
Explain the difference between attribution models in marketing analytics

## 🎯 Pattern 7: Follow-up Questions

Let AI generate insightful follow-up questions:

In [ ]:
%%calliope followup-questions marketing
Based on our marketing campaign data, what questions should I be asking?

## 🔗 Pattern 8: Cross-Analysis with Python + AI

In [ ]:
# Query the data
%%calliope run-sql marketing
SELECT channel, AVG(clicks * 1.0 / impressions * 100) as avg_ctr
FROM metrics m
JOIN campaigns c ON m.campaign_id = c.campaign_id
GROUP BY channel

In [ ]:
# Store results in DataFrame for further analysis
import sqlite3
conn = sqlite3.connect('marketing_data.db')
df_results = pd.read_sql_query("""
    SELECT 
        c.channel,
        DATE(m.date) as date,
        SUM(m.clicks) as total_clicks,
        SUM(m.impressions) as total_impressions,
        SUM(m.conversions) as total_conversions
    FROM metrics m
    JOIN campaigns c ON m.campaign_id = c.campaign_id
    GROUP BY c.channel, DATE(m.date)
    ORDER BY date
""", conn)
conn.close()

print(df_results.head())

In [ ]:
%%ai gemini:gemini-2.5-flash --format code
Using the df_results DataFrame, create a time-series visualization with plotly 
showing conversion trends for each channel over time

## 🚀 Pattern 9: Model Comparison

Compare how different models handle the same question:

In [ ]:
# Using OpenAI
%%calliope ask-sql marketing --sql-model openai
Find underperforming campaigns that should be paused or optimized

In [ ]:
# Using Anthropic (typically better for complex analytical queries)
%%calliope ask-sql marketing --sql-model anthropic
Find underperforming campaigns that should be paused or optimized

In [ ]:
# Using Gemini (typically faster)
%%calliope ask-sql marketing --sql-model gemini
Find underperforming campaigns that should be paused or optimized

## 🎓 Pattern 10: Learning & Documentation

In [ ]:
# Generate SQL to learn
%%calliope generate-sql marketing --sql-model anthropic
Calculate customer acquisition cost (CAC) by channel

In [ ]:
# Ask AI to explain the SQL
%%ai anthropic-chat:claude-3-5-sonnet-20241022
Explain the SQL query above line by line. What is customer acquisition cost and why is it important?

## 💡 Best Practices

### Model Selection
- **OpenAI (GPT-4o)**: General-purpose, reliable for most queries
- **Anthropic (Claude)**: Best for complex analytical SQL and nuanced reasoning
- **Gemini**: Fast, good for simple queries and code generation
- **Cohere**: Excellent for structured data analysis
- **Ollama**: Local deployment, privacy-focused

### Workflow Tips
1. **Query First**: Use Calliope to get data, then AI to analyze
2. **Iterate**: Generate SQL → Review → Modify → Execute
3. **Multi-Model**: Use different models for different strengths
4. **Document**: Use AI to explain complex queries
5. **Automate**: Build pipelines combining queries and AI analysis

### Performance
- Use `--sql-model anthropic` for complex queries
- Use `gemini` for speed
- Use `--to-ai` flag for enhanced insights
- Cache results in DataFrames for repeated analysis

## 🎯 Key Takeaways

- Calliope wraps Jupyter AI, giving you full access to all providers
- Combine database queries with AI analysis for powerful insights
- Use the right model for the right task
- Build automated pipelines: Query → Analyze → Report
- Iterate and refine with multiple AI models

## 🔧 Common Commands Summary

```python
# Calliope database commands
%%calliope ask-sql datasource_id
%%calliope generate-sql datasource_id --sql-model anthropic
%%calliope run-sql datasource_id
%%calliope ask datasource_id --model gpt4

# Jupyter AI commands (also available in Calliope)
%%ai anthropic-chat:claude-3-5-sonnet-20241022
%%ai openai-chat:gpt-4o --format code
%%ai gemini:gemini-2.5-flash --format code
```

## 📚 Next Steps

- **Calliope-RAGTraining.ipynb** - Train Calliope on your business logic
- **Calliope-AdvancedQueries.ipynb** - Complex analytical patterns
- Experiment with different model combinations
- Build your own automated analysis pipelines